In [ ]:
library(moments)
library(dplyr)
library(tidyr)
library(lubridate)
library(knitr)
library(kableExtra)


# ---- 1. Load panel --------------------------------------------------------
panel <- read.csv("../data/monthly_panel.csv")
raw_df <- read.csv("../data/sc_control_data_final_with_agg_score_and_vuln_count.csv")


raw_df <- raw_df %>%
    mutate(year_month = floor_date(as_date(published_at), "month"))

In [ ]:
raw_metrics <- c(
  "Aggregate_score", "Binary.Artifacts_score", "Code.Review_score", 
  "Contrib_score","Dangerous.Workflow_score", "DependUpTool_score", 
  "Fuzzing_score", "License_score",
  "Maintained_score", "Pinned.Dependencies_score",
  "Security.Policy_score",
  "Token.Permissions_score"
)

In [ ]:

within_var_summary <- lapply(raw_metrics, function(m) {
  raw_df %>%
    group_by(package_name) %>%
    summarise(
      n_obs = n(),
      has_any_valid = any(!is.na(.data[[m]]) & .data[[m]] != -1),
      n_distinct_valid = n_distinct(.data[[m]][!is.na(.data[[m]]) & .data[[m]] != -1]),
      all_zero = has_any_valid && all(.data[[m]][!is.na(.data[[m]]) & .data[[m]] != -1] == 0),
      all_ten  = has_any_valid && all(.data[[m]][!is.na(.data[[m]]) & .data[[m]] != -1] == 10),
      all_mid  = has_any_valid && n_distinct_valid == 1 && !all_zero && !all_ten,
      fluctuates = has_any_valid && n_distinct_valid > 1,
      .groups = "drop"
    ) %>%
    summarise(
      metric = m,
      n_packages = n(),
      pct_missing_pkg = round(mean(!has_any_valid) * 100, 1),
      only_eq_0   = round(mean(all_zero) * 100, 1),
      only_eq_1_9 = round(mean(all_mid) * 100, 1),
      only_eq_10  = round(mean(all_ten) * 100, 1),
      perc_variation = round(mean(fluctuates) * 100, 1)
    )
}) %>%
  bind_rows() %>%
  arrange(desc(perc_variation))

# Sanity check: all 5 categories should now sum to 100
sanity_checks <- within_var_summary %>%
  transmute(
    metric,
    check_total = pct_missing_pkg + only_eq_0 + only_eq_1_9 + only_eq_10 + perc_variation,
    sums_to_100 = abs(check_total - 100) < 0.2
  )
print(sanity_checks, width = Inf)
print(within_var_summary, width = Inf)

In [ ]:
latex_df <- within_var_summary %>%
  rename(
    `Metric` = metric,
    `\\% -1/Missing` = pct_missing_pkg,
    `\\% = 0` = only_eq_0,
    `\\% = 1 - 9` = only_eq_1_9,
    `\\% = 10` = only_eq_10,
    `\\% w/ Fluctuations` = perc_variation
  ) %>%
  select(`Metric`, `\\% -1/Missing`, `\\% = 0`, `\\% = 1 - 9`, `\\% = 10`, `\\% w/ Fluctuations`)

latex_table <- kable(
  latex_df,
  format = "latex",
  booktabs = TRUE,
  escape = FALSE,
  align = "lccccc",
  caption = "\\textsc{Within-Package Variation in Scorecard Metrics (Grouped by Package)}"
)

# Wrap in resizebox so the whole table scales to fit the column width
latex_table <- sub("\\\\begin\\{tabular\\}", "\\\\resizebox{\\\\linewidth}{!}{\\\\begin{tabular}", latex_table)
latex_table <- sub("\\\\end\\{tabular\\}", "\\\\end{tabular}}", latex_table)

cat(latex_table)

In [ ]:
print("Before transformation:")
skewness(panel$loc, na.rm = TRUE)
skewness(panel$dependents, na.rm = TRUE)
skewness(panel$downloads_7_day_total, na.rm = TRUE)
skewness(panel$version_age_days, na.rm = TRUE)
skewness(panel$Release_count)

print("After log transformation:")
## log (x + 1) to handle the skewness of LOC, dependents, download count, and release count
skewness(log1p(panel$loc), na.rm = TRUE)
skewness(log1p(panel$dependents), na.rm = TRUE)
skewness(log1p(panel$downloads_7_day_total), na.rm = TRUE)
skewness(log1p(panel$Release_count))
